<a href="https://colab.research.google.com/github/pankajkumar8709/Machine-Learning-projects/blob/main/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XGBoost — Revision Notes

## 1. What is XGBoost?

**XGBoost** (eXtreme Gradient Boosting) is a scalable, regularized implementation of **gradient boosted decision trees (GBDT)**. It builds an ensemble of decision trees sequentially, where each new tree corrects the errors made by the previous ensemble.

Key traits:
- Supervised learning algorithm (regression, classification, ranking).
- Ensemble method: combines many **weak learners** (shallow trees) into one **strong learner**.
- Uses **gradient boosting** + **second-order Taylor approximation** + **regularization** for speed and accuracy.

---

## 2. Core Idea — Boosting

Boosting builds models **sequentially**, where each model tries to fix the mistakes of the combined previous models.

$$
\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i)
$$

- $\hat{y}_i^{(t)}$: prediction at boosting round $t$
- $f_t$: the new tree added at round $t$
- We keep adding trees $f_1, f_2, \dots, f_T$ until the ensemble prediction is good enough.

Final prediction:

$$
\hat{y}_i = \sum_{t=1}^{T} f_t(x_i), \quad f_t \in \mathcal{F}
$$

where $\mathcal{F}$ is the space of regression trees.

---

## 3. Mathematical Intuition

### 3.1 Objective Function

XGBoost minimizes an objective made of two parts:

$$
\mathcal{L}^{(t)} = \sum_{i=1}^{n} l(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)
$$

- **Loss term** $l(y_i, \hat{y}_i)$: measures how far predictions are from actual values (e.g., squared error, log loss).
- **Regularization term** $\Omega(f_t)$: penalizes model complexity to avoid overfitting.

$$
\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2
$$

- $T$ = number of leaves in the tree
- $w_j$ = weight (output score) of leaf $j$
- $\gamma$ = penalty per leaf (controls tree size)
- $\lambda$ = L2 regularization on leaf weights

### 3.2 Second-Order Taylor Expansion (the key XGBoost trick)

Instead of only using the gradient (like standard gradient boosting), XGBoost approximates the loss using a **second-order Taylor expansion** around the current prediction:

$$
\mathcal{L}^{(t)} \approx \sum_{i=1}^{n} \left[ l(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2 \right] + \Omega(f_t)
$$

where:
- $g_i = \partial_{\hat{y}} l(y_i, \hat{y}_i^{(t-1)})$ → **first derivative (gradient)**
- $h_i = \partial^2_{\hat{y}} l(y_i, \hat{y}_i^{(t-1)})$ → **second derivative (Hessian)**

Using both gradient and Hessian gives more accurate, faster-converging updates than using gradient alone (as in classic GBM).

### 3.3 Optimal Leaf Weight

For a fixed tree structure, the loss is minimized when each leaf's weight is:

$$
w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
$$

where $I_j$ is the set of data points falling into leaf $j$.

Plugging this back gives the **optimal objective value** for a tree structure $q$:

$$
\tilde{\mathcal{L}}^{(t)}(q) = -\frac{1}{2} \sum_{j=1}^{T} \frac{\left(\sum_{i \in I_j} g_i\right)^2}{\sum_{i \in I_j} h_i + \lambda} + \gamma T
$$

This is used as a **scoring function** to evaluate how good a tree structure is — lower is better.

### 3.4 Split Finding (Gain Formula)

When deciding whether to split a node into left (L) and right (R) children, XGBoost computes the **Gain**:

$$
\text{Gain} = \frac{1}{2}\left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma
$$

- $G_L, G_R$: sum of gradients in left/right nodes
- $H_L, H_R$: sum of Hessians in left/right nodes
- The first two terms = benefit of splitting; the third term = loss if not split; $\gamma$ = complexity penalty for adding a split.

**A split is only made if Gain > 0** — this is how XGBoost naturally performs pruning.

### 3.5 Shrinkage and Column Subsampling

- **Shrinkage (learning rate, $\eta$):** scales the contribution of each tree, $\hat{y}^{(t)} = \hat{y}^{(t-1)} + \eta \cdot f_t(x)$, so no single tree dominates — improves generalization.
- **Column (feature) subsampling:** similar to Random Forest, reduces correlation between trees and speeds up training.

---

## 4. Why XGBoost is "Extreme" / Fast

- **Second-order optimization** → faster, more accurate convergence.
- **Regularization (L1 & L2)** built into the objective → less overfitting than plain GBM.
- **Approximate split-finding algorithm** (weighted quantile sketch) → handles large datasets efficiently.
- **Sparsity-aware split finding** → handles missing values automatically by learning a default split direction.
- **Parallelized tree construction** → feature values are pre-sorted and split-search is parallelized (not the trees themselves, since boosting is sequential).
- **Cache-aware access & out-of-core computation** → efficient for very large datasets.
- **Built-in cross-validation** and early stopping support.

---

## 5. Hyperparameters

### 5.1 General Parameters
| Parameter | Description |
|---|---|
| `booster` | Type of model: `gbtree` (default), `gblinear`, `dart` |
| `n_jobs` / `nthread` | Number of parallel threads |
| `verbosity` | Logging level |

### 5.2 Tree Booster Parameters (control model complexity)
| Parameter | Description | Typical Range |
|---|---|---|
| `n_estimators` | Number of boosting rounds (trees) | 100–1000 |
| `max_depth` | Maximum depth of a tree — controls overfitting | 3–10 |
| `min_child_weight` | Minimum sum of Hessian (instance weight) needed in a child — higher = more conservative | 1–10 |
| `gamma` (`min_split_loss`) | Minimum loss reduction required to make a split (the $\gamma$ above) | 0–5 |
| `subsample` | Fraction of rows sampled per tree | 0.5–1.0 |
| `colsample_bytree` | Fraction of features sampled per tree | 0.5–1.0 |
| `colsample_bylevel` / `colsample_bynode` | Feature sampling per level/split | 0.5–1.0 |
| `lambda` (`reg_lambda`) | L2 regularization on leaf weights | 0–10 |
| `alpha` (`reg_alpha`) | L1 regularization on leaf weights (encourages sparsity) | 0–10 |
| `max_leaves` | Max number of leaves (used with `grow_policy=lossguide`) | — |
| `tree_method` | Split algorithm: `auto`, `exact`, `approx`, `hist`, `gpu_hist` | — |

### 5.3 Learning Task Parameters
| Parameter | Description |
|---|---|
| `eta` (`learning_rate`) | Shrinkage factor applied to each tree's output (0.01–0.3 typical) |
| `objective` | Loss function: `reg:squarederror`, `binary:logistic`, `multi:softmax`, `rank:pairwise`, etc. |
| `eval_metric` | Metric for validation: `rmse`, `mae`, `logloss`, `auc`, `error`, etc. |
| `scale_pos_weight` | Balances positive/negative classes in imbalanced classification |
| `early_stopping_rounds` | Stop training if validation metric doesn't improve for N rounds |

### 5.4 Practical Tuning Tips
- Start with `max_depth` (3–6), `eta` (0.05–0.1), `n_estimators` (moderate, use early stopping).
- Lower `eta` → increase `n_estimators` proportionally.
- Use `subsample` and `colsample_bytree` < 1 to reduce overfitting.
- Increase `gamma`, `min_child_weight`, `lambda`, `alpha` to regularize more.
- Use `scale_pos_weight` for imbalanced classification instead of oversampling.
- Tune via grid/random search or Bayesian optimization (e.g., Optuna), using cross-validation.

---

## 6. Which Datasets XGBoost Works Best On

**XGBoost excels on:**
- **Structured / tabular data** — its natural strength (numeric + categorical features in rows/columns).
- **Medium-to-large datasets** (thousands to millions of rows) — enough data for trees to learn meaningful splits.
- Data with **non-linear relationships and feature interactions** that linear models miss.
- Data with **missing values** — handled natively without imputation.
- **Classification, regression, and ranking** problems: fraud detection, credit scoring, churn prediction, click-through-rate prediction, sales/demand forecasting, medical risk scoring, search ranking (learning-to-rank).
- Competitions like **Kaggle** — historically dominant on tabular-data leaderboards.

**XGBoost is not ideal for:**
- **Unstructured data**: images, audio, raw text, video — deep learning (CNNs, Transformers) performs far better here.
- **Very high-dimensional sparse data** with extremely large feature spaces (e.g., huge one-hot encoded NLP data) — linear models or neural nets may generalize better and train faster.
- **Very small datasets** — trees can overfit; simpler models (linear/logistic regression) may generalize better.
- **Streaming / online learning** scenarios requiring continuous incremental updates — XGBoost is primarily a batch learner (though incremental training is possible to a limited extent).
- Problems requiring **extrapolation** beyond the training data range — tree-based models predict poorly outside the range of values seen in training (unlike linear models).

---

## 7. Quick Comparison: XGBoost vs Random Forest vs GBM

| Aspect | Random Forest | Classic GBM | XGBoost |
|---|---|---|---|
| Tree building | Parallel, independent trees | Sequential | Sequential |
| Bias/Variance | Reduces variance (bagging) | Reduces bias (boosting) | Reduces bias + regularized |
| Optimization | N/A | First-order gradient | Second-order (gradient + Hessian) |
| Regularization | Minimal | Minimal | L1 + L2 + tree pruning built-in |
| Speed | Fast (parallel trees) | Slower | Optimized (parallel split-finding, histogram methods) |
| Missing values | Needs imputation | Needs imputation | Handled natively |

---

## 8. One-Line Summary

> XGBoost = Gradient Boosted Trees + second-order Taylor approximation of the loss + explicit L1/L2 regularization + engineering optimizations (sparsity-awareness, parallel split-finding, cache efficiency) → a fast, accurate, and regularized boosting algorithm that dominates on structured/tabular datasets.